# The 95 GeV excess: what the SM has to say about a second scalar

Three searches see small excesses at similar masses: CMS diphoton at 95.4 GeV, CMS ditau at
95 to 100 GeV and LEP $b\bar b$ near 98 GeV. None is significant on its own, and ATLAS sees much
less in diphoton. This notebook climbs the first rungs of the ladder for this record (A2):

- it proves, with the SM taken from feynlag-models, that the SM has room for exactly **one**
  physical scalar, so it predicts no resonant signal at 95 GeV;
- it puts the ditau and diphoton channels on **one scale**, a signal strength $\mu$, so that a
  candidate model can later be confronted with both;
- it finds, with feynlag, the lowest operators through which a new singlet scalar reaches the SM;
- it builds two candidate models with a scalar at 95 GeV and checks what they give back.

It does not fit anything and it does not claim the channels agree; that is A3 work.

Each section runs four moves: **set up**, **collect**, **recognise**, **check**. Before the cell
that settles each question, a blockquote asks you to commit to an expectation.

In [1]:
import math
import sys
from pathlib import Path

import sympy as sp
from IPython.display import Markdown, Math, display

# nbclient/nbmake set the kernel's cwd to this notebook's own directory; make sure it (and the
# repo root two levels up) are importable regardless of how the kernel was launched.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parents[1]
for p in (NB_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from feynlag_anomalies.latex import latex
from feynlag_anomalies.registry import load as load_anomaly
from anomalies.higgs_95gev import checkpoints_def as cd

sp.init_printing(use_latex="mathjax", latex_printer=latex)


def ok(msg):
    print("✓", msg)


anomaly = load_anomaly("higgs_95gev")
obs = {o.name: o for o in anomaly.observables}
print(f"{anomaly.title}: status {anomaly.status} ({anomaly.status_date}), maturity {anomaly.maturity}")

~95 GeV diphoton/ditau/bb excess: status hint (2026-10-15), maturity A2


---
## 0. What is measured: three small excesses, two different units

The record holds what each experiment printed, read from the papers and CMS's own supplementary
material *(cited, see `anomaly.yaml` `sources`)*.

> **Before running the next cell.** CMS diphoton prints a signal strength $\mu$, a pure number.
> CMS ditau prints $\sigma \times B$ in pb. Can you compare $\mu_{\gamma\gamma} = 0.33$ with
> $\sigma B_{\tau\tau} = 7.8$ pb? What extra number would you need?

In [2]:
# ---- MOVE 1: set up.  The observables, straight from anomaly.yaml. ----
fmt = lambda x: x if isinstance(x, str) else f"{x:.4g}"
rows = ["| observable | value | ± (stat. col.) | ± (sys. col.) | units |", "|---|---|---|---|---|"]
rows += [f"| {o.name} | {fmt(o.value)} | {fmt(o.stat_uncertainty)} | "
         f"{'—' if o.sys_uncertainty is None else fmt(o.sys_uncertainty)} | {o.units} |"
         for o in anomaly.observables]
display(Markdown("\n".join(rows)))
sig = anomaly.significance
print(f"diphoton significance: {sig.local} sigma local, {sig.global_} sigma global")
ditau_units = obs["sigma(gg -> phi) x B(phi -> tau tau) at m_phi = 95 GeV (CMS ditau)"].units
gg_units = obs["signal strength mu at 95.4 GeV (CMS diphoton, best fit)"].units
assert ditau_units == "pb" and gg_units.startswith("ratio")
ok("the channels are recorded in different units: pb for ditau, a ratio for diphoton")

| observable | value | ± (stat. col.) | ± (sys. col.) | units |
|---|---|---|---|---|
| sigma(gg -> phi) x B(phi -> tau tau) at m_phi = 95 GeV (CMS ditau) | 7.8 | 3.5 | — | pb |
| signal strength mu at 95.4 GeV (CMS diphoton, best fit) | 0.33 | 0.155 | — | ratio to SM-like Higgs cross section |
| signal strength mu_tautau at 95 GeV (CMS ditau, normalised to SM-like ggF x B) | 1.374 | 0.616 | 0.169 | ratio to SM-like Higgs ggF cross section x B(tau tau) |
| signal strength mu near 98 GeV (LEP bb-bar, best fit) | TODO_VERIFY | TODO_VERIFY | — | ratio to SM Higgs cross section |

diphoton significance: 2.9 sigma local, 1.3 sigma global
✓ the channels are recorded in different units: pb for ditau, a ratio for diphoton


**Read it.** The diphoton number is already a ratio to an SM-like Higgs boson of the same mass;
the ditau one is a cross section. They cannot be compared until the ditau number is divided by
what an SM-like Higgs boson at 95 GeV would give in the same channel. That denominator is the
subject of section 2. The fourth row is the result of section 2, stored in the record and
recomputed below; the LEP row is still `TODO_VERIFY`, because the LEP paper prints no signal
strength.

First, the question that makes this an anomaly: is there any room in the SM for a scalar at
95 GeV?

---
## 1. The SM has one scalar, and it is not at 95 GeV

The SM's scalar sector is one complex doublet $H = (G^+, H^0)$. The model comes from
feynlag-models (`sm`, maturity L3) through the registry; nothing is declared here.

> **Before running the next cells.** A complex doublet has four real fields. After electroweak
> symmetry breaking, $W^\pm$ and $Z$ become massive. How many physical scalars are left, and is
> there any free choice in that number?

In [3]:
# ---- MOVE 1: set up.  The SM from the registry, and its doublet. ----
from feynlag_models.registry import build

sm = build("sm")
ew = sm.extra["ew"]
display(Math(r"H = \begin{pmatrix}" + r"\\".join(latex(c) for c in ew.H.components) + r"\end{pmatrix}"))
n_dof = 2 * len(ew.H.components)
print(f"real degrees of freedom in H: {n_dof}")

<IPython.core.display.Math object>

real degrees of freedom in H: 4


In [4]:
# ---- MOVE 2: collect.  The model's own scalar mass matrices after EWSB. ----
b = sm.bosons
M_neutral = sm.model.mass_matrix([b["h"], b["G0"]])
M_charged = sm.model.mass_matrix([b["Gp"]], charged=True)
display(Math(r"M^2_{(" + latex(b["h"]) + "," + latex(b["G0"]) + r")} = " + latex(M_neutral)))
display(Math(r"M^2_{(" + latex(b["Gp"]) + r")} = " + latex(M_charged)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**Recognise.** The neutral block is diagonal: one eigenvalue $2\lambda v^2$, one zero. The
charged block is zero. The zeros are the would-be Goldstones $G^0$, $G^\pm$, three real fields,
which become the longitudinal $Z$ and $W^\pm$. So $4 - 3 = 1$ physical scalar, $h$, with
$m_h^2 = 2\lambda v^2$ *(computed; the same result is the model's key result
`test_tadpole_and_higgs_mass` in feynlag-models)*. There is no free choice: the count is fixed
by the field content, and only the mass, through $\lambda$, is an input.

In [5]:
# ---- MOVE 4: check.  Count eigenvalues, against the checkpoint. ----
check_1 = cd.make_check_1(sm)
eig_n = [e for e, m in M_neutral.eigenvals().items() for _ in range(m)]
eig_c = [e for e, m in M_charged.eigenvals().items() for _ in range(m)]
massive = [e for e in eig_n if sp.simplify(e) != 0]
n_goldstone = (len(eig_n) - len(massive)) + 2 * sum(1 for e in eig_c if sp.simplify(e) == 0)
answer_1 = len(massive)
assert n_goldstone == len(sm.goldstones) == 3
assert n_dof - n_goldstone == answer_1
assert sp.simplify(massive[0] - 2 * ew.lam.s * ew.v.s**2) == 0
assert check_1(answer_1)
ok("one physical neutral scalar, m_h^2 = 2 lambda v^2, three Goldstones eaten")

[step_1] correct -- got 1
✓ one physical neutral scalar, m_h^2 = 2 lambda v^2, three Goldstones eaten


Now put in numbers. The model's benchmark sets $\lambda$ from its input $m_h$; the count
above says that this one input is all the SM has. The one scalar is the observed Higgs boson,
whose measured mass is in the record *(cited: PDG 2024)*.

> **Before running the next cell.** With one scalar whose mass is chosen to be the observed
> Higgs boson, what does the SM predict for a resonance at 95 GeV?

In [6]:
# ---- MOVE 4: check.  Every physical SM scalar at the benchmark, against the 95 GeV window. ----
values = sm.values()
masses = [math.sqrt(float(m2.subs(values))) for m2 in massive]
lo, hi = cd.WINDOW_GEV   # the mass window of the excesses (CMS gamma gamma, CMS tau tau, LEP)
in_window = [m for m in masses if lo <= m <= hi]
ref = anomaly.sm_prediction.sm_reference_values
m_H, dm_H = ref["m_H_measured_GeV"], ref["m_H_measured_unc_GeV"]
print(f"SM scalar masses at the benchmark: {[f'{m:.2f} GeV' for m in masses]} (input MH = {sm.benchmark['MH']})")
print(f"measured Higgs mass (PDG 2024): {m_H} +- {dm_H} GeV")
assert in_window == []
assert abs(masses[0] - m_H) < 1.0 and m_H - 5 * dm_H > hi
ok(f"no SM scalar in {lo:.0f}-{hi:.0f} GeV: the one scalar is the measured {m_H} GeV boson")

SM scalar masses at the benchmark: ['125.00 GeV'] (input MH = 125.0)
measured Higgs mass (PDG 2024): 125.2 +- 0.11 GeV
✓ no SM scalar in 90-100 GeV: the one scalar is the measured 125.2 GeV boson


**A trap worth a cell.** The SM *does* have a state in that window: the $Z$ boson.

In [7]:
# ---- peek: the Z mass at the benchmark. ----
m_Z = sm.params.numeric()[sm.extra["MZ"].symbol]
display(Math(r"m_Z = " + latex(sm.extra["MZ"].expr) + rf" = {m_Z:.2f}\ \text{{GeV}}"))

<IPython.core.display.Math object>

The $Z$ sits at 91 GeV, inside the window, but it is spin 1. A spin-1 particle cannot decay to
two photons (the Landau–Yang theorem) *(cited: L. D. Landau, Dokl. Akad. Nauk SSSR 60 (1948)
207; C. N. Yang, Phys. Rev. 77 (1950) 242)*, so it cannot be the diphoton peak. In the ditau
and $b\bar b$ channels it is a known background, already part of the SM expectation the
experiments model *(physics judgment)*.

**The judgment to keep:** the SM's prediction at 95 GeV is a *structural* zero. It comes from
the field content, one doublet, not from a parameter choice. The protection is `field_content`
*(see [`docs/protections.md`](../../docs/protections.md))*: a second scalar needs a new field.
That is why a real resonance here would be new physics, and why the minimal models to try add
one scalar field (a singlet or a second doublet).

---
## 2. Ditau and diphoton on one scale

A signal strength divides what is seen by what an SM-like Higgs boson of the same mass would
give in the same channel:

$$\mu_{\tau\tau} = \frac{[\sigma(gg\to\phi)\,B(\phi\to\tau\tau)]_{\rm obs}}
{\sigma_{\rm ggF}^{\rm SM\text{-}like}(95\ \text{GeV}, 13\ \text{TeV})\;
B^{\rm SM\text{-}like}(H\to\tau\tau, 95\ \text{GeV})} .$$

"SM-like" means a hypothetical Higgs boson with SM couplings and a mass of 95 GeV. Its numbers
are tabulated by the LHC Higgs Working Group *(cited, stored in
`sm_prediction.sm_reference_values`)*.

> **Before running the next cells.** A 95 GeV SM-like Higgs boson is produced more often than
> the 125 GeV one (gluon luminosity is larger at lower mass). Is $\mu_{\tau\tau}$ then larger or
> smaller than 1, given $\sigma B_{\rm obs} = 7.8$ pb? Guess the size of the denominator first.

In [8]:
# ---- MOVE 1: set up.  The SM-like inputs and their sources, from the record. ----
ref = anomaly.sm_prediction.sm_reference_values
rows = ["| input | value |", "|---|---|"] + [f"| `{k}` | {v:g} |" for k, v in ref.items()]
display(Markdown("\n".join(rows)))
for s in anomaly.sources:
    if "LHC Higgs WG" in s.identifier:
        print(f"source: {s.identifier.split(' (')[0]} -- consulted {s.consulted_on}")

| input | value |
|---|---|
| `sigma_ggF_95GeV_13TeV_pb` | 68.25 |
| `sigma_ggF_95GeV_13TeV_scale_up_pct` | 8.6 |
| `sigma_ggF_95GeV_13TeV_scale_down_pct` | 8.9 |
| `sigma_ggF_95GeV_13TeV_pdf_alphas_pct` | 3.3 |
| `BR_tautau_95GeV` | 0.0832 |
| `BR_tautau_95GeV_up_pct` | 6.94 |
| `BR_tautau_95GeV_down_pct` | 6.77 |
| `sigma_ggF_100GeV_13TeV_EW_factor` | 1.0418 |
| `m_H_measured_GeV` | 125.2 |
| `m_H_measured_unc_GeV` | 0.11 |

source: LHC Higgs WG twiki LHCPhysics/CERNYellowReportPageBSMAt13TeV -- consulted 2026-10-16
source: LHC Higgs WG twiki LHCPhysics/CERNYellowReportPageBR2014 -- consulted 2026-10-16


In [9]:
# ---- MOVE 2: collect.  The ratio, by hand from the record. ----
ditau = obs["sigma(gg -> phi) x B(phi -> tau tau) at m_phi = 95 GeV (CMS ditau)"]
denom = ref["sigma_ggF_95GeV_13TeV_pb"] * ref["BR_tautau_95GeV"]
answer_2 = ditau.value / denom
exp_err = ditau.stat_uncertainty / denom
print(f"SM-like sigma_ggF x B(tau tau) at 95 GeV = {denom:.3f} pb")
print(f"mu_tautau = {ditau.value} pb / {denom:.3f} pb = {answer_2:.3f} +- {exp_err:.3f} (exp)")

SM-like sigma_ggF x B(tau tau) at 95 GeV = 5.678 pb
mu_tautau = 7.8 pb / 5.678 pb = 1.374 +- 0.616 (exp)


**Recognise.** The denominator is about 5.7 pb, so 7.8 pb is a signal strength a little above 1.
The experimental error, the symmetrised CMS error propagated linearly, is about 45 % of the value.

The theory error on the denominator is separate: the ggF scale variation, its PDF+$\alpha_s$
error, the $B(\tau\tau)$ error and a missing NLO electroweak correction, each symmetrised and
added in quadrature *(decision, see `decisions.md`)*. The ggF table gives the EW factor only from
100 GeV up, so it is not applied at 95 GeV; its 100 GeV size stands in as an uncertainty. It is kept apart from the experimental error so that a later fit can treat it
as a correlated nuisance.

In [10]:
# ---- MOVE 4: check.  Against the checkpoint and the row stored in anomaly.yaml. ----
check_2 = cd.make_check_2(anomaly)
assert check_2(answer_2)
scale = (ref["sigma_ggF_95GeV_13TeV_scale_up_pct"] + ref["sigma_ggF_95GeV_13TeV_scale_down_pct"]) / 2
br = (ref["BR_tautau_95GeV_up_pct"] + ref["BR_tautau_95GeV_down_pct"]) / 2
ew = (ref["sigma_ggF_100GeV_13TeV_EW_factor"] - 1) * 100
th_err = answer_2 * math.sqrt(scale**2 + ref["sigma_ggF_95GeV_13TeV_pdf_alphas_pct"] ** 2 + br**2 + ew**2) / 100
row = obs["signal strength mu_tautau at 95 GeV (CMS ditau, normalised to SM-like ggF x B)"]
assert abs(row.value - answer_2) < 5e-4 and abs(row.stat_uncertainty - exp_err) < 5e-4
assert abs(row.sys_uncertainty - th_err) < 5e-4
ok(f"mu_tautau = {answer_2:.2f} +- {exp_err:.2f} (exp) +- {th_err:.2f} (theory) matches the record")

[step_2] correct -- got 1.37363
✓ mu_tautau = 1.37 +- 0.62 (exp) +- 0.17 (theory) matches the record


Now the two signal strengths side by side.

> **Before running the next cell.** Both numbers are now ratios to an SM-like Higgs boson. Name
> one reason why they need not be equal even for a single new scalar.

In [11]:
# ---- MOVE 3: recognise.  Side by side, without a verdict. ----
gg = obs["signal strength mu at 95.4 GeV (CMS diphoton, best fit)"]
display(Markdown("\n".join([
    "| channel | production | $\\mu$ | ± exp. | ± theory |", "|---|---|---|---|---|",
    f"| CMS $\\gamma\\gamma$, 95.4 GeV | all modes | {gg.value:.2f} | {gg.stat_uncertainty:.3f} (symmetrised) | — |",
    f"| CMS $\\tau\\tau$, 95 GeV | $gg\\to\\phi$ | {answer_2:.2f} | {exp_err:.2f} | {th_err:.2f} |",
])))

| channel | production | $\mu$ | ± exp. | ± theory |
|---|---|---|---|---|
| CMS $\gamma\gamma$, 95.4 GeV | all modes | 0.33 | 0.155 (symmetrised) | — |
| CMS $\tau\tau$, 95 GeV | $gg\to\phi$ | 1.37 | 0.62 | 0.17 |

**Read it, and stop there.** These are two numbers on one scale; nothing here tests whether they
are compatible. That needs a likelihood with correlated theory errors and the asymmetric
experimental errors, which is the stage-1 fit at A3.

**Caveats** that any comparison must carry:

- **Production differs.** Ditau is gluon fusion only; the diphoton $\mu$ is for the combined
  production modes. A model with non-SM ratios of production modes moves them differently.
- **Decay differs.** A scalar whose couplings are not a common rescaling of the SM ones changes
  $B(\gamma\gamma)$ and $B(\tau\tau)$ differently, so $\mu_{\gamma\gamma} \neq \mu_{\tau\tau}$
  is allowed in principle.
- **The masses are not identical.** Ditau peaks at 100 GeV (3.1σ local) and is evaluated here at
  95 GeV; diphoton is at 95.4 GeV.
- **Never fit both ditau rows.** The derived $\mu_{\tau\tau}$ row and the $\sigma B$ row
  are one measurement; a fit takes one or the other.
- **Inputs from two reports.** The cross section is from CERN Report 4, the branching ratio from
  CERN Report 3, the only LHC Higgs WG table with an SM-like $B$ at 95 GeV. A cross-check from
  the Report 4 partial widths at 95 GeV agrees to about 1 % *(computed, see `decisions.md`)*.

**The judgment to keep:** a channel only enters a fit once it is on the same scale as the others.
Converting a cross section into a signal strength makes it comparable, and it also brings in a
theory error that comes from the SM-like denominator, not from the experiment.

---
## 3. A singlet talks to the SM only through $H^\dagger H$

Section 1 showed that a second scalar needs a new field. The cheapest is a real gauge singlet $S$.
The fields come from feynlag-models' `sm_singlet_z2` (L3) through the registry; feynlag then
lists every gauge-invariant term they allow. Nothing is typed by hand.

> **Before running the next cells.** $S$ carries no charge at all. List the terms of dimension
> $\le 4$ that couple it to the SM. Can it couple to a pair of SM fermions?

In [12]:
# ---- MOVE 1: set up.  The singlet model's fields, from the registry. ----
from anomalies.higgs_95gev.solutions import step_3

singlet = step_3.build_singlet()
H, S = step_3.fields(singlet)
print("scalars:", H.name, "(doublet, Y = 1/2) and", S.name, "(real singlet)")
print("gauge groups:", ", ".join(g.name for g in singlet.pieces.gauge_groups))

scalars: H (doublet, Y = 1/2) and S (real singlet)
gauge groups: SU2L, U1Y, SU3c


In [13]:
# ---- MOVE 2: collect.  feynlag's invariant potential up to dim 4, without and with the Z2. ----
terms = step_3.potential_terms(singlet)
terms_z2 = step_3.potential_terms(singlet, with_z2=True)
z2_labels = {t.label for t in terms_z2}
rows = ["| term | dim | couples S to H | survives the $Z_2$ |", "|---|---|---|---|"]
rows += [f"| ${latex(t.expr)}$ | {t.dim} | {'yes' if step_3.couples_s_to_sm(t, singlet) else ''} | "
         f"{'yes' if t.label in z2_labels else ''} |" for t in terms]
display(Markdown("\n".join(rows)))
yuk = step_3.yukawa_terms(singlet)
print(f"Yukawa terms with [H, S]: {len(yuk)}, with [H] alone: "
      f"{len(step_3.yukawa_terms(singlet, with_singlet=False))}; terms containing S: "
      f"{len(step_3.singlet_fermion_terms(singlet))}")

| term | dim | couples S to H | survives the $Z_2$ |
|---|---|---|---|
| $G^+ G^- + H^0 \overline{H^0}$ | 2 |  | yes |
| $S^{2}$ | 2 |  | yes |
| $G^+ S G^- + H^0 S \overline{H^0}$ | 3 | yes |  |
| $\left(G^+\right)^{2} \left(G^-\right)^{2} + 2 G^+ H^0 G^- \overline{H^0} + \left(H^0\right)^{2} \overline{H^0}^{2}$ | 4 |  | yes |
| $S^{3}$ | 3 |  |  |
| $G^+ S^{2} G^- + H^0 S^{2} \overline{H^0}$ | 4 | yes | yes |
| $S^{4}$ | 4 |  | yes |

Yukawa terms with [H, S]: 3, with [H] alone: 3; terms containing S: 0


**Recognise.** Two terms couple $S$ to the SM: $S\,H^\dagger H$ (dimension 3) and
$S^2 H^\dagger H$ (dimension 4), the **Higgs portals** *(computed by feynlag)*. Adding $S$ to the
Yukawa scan changes nothing: no pair of SM fermions is a gauge singlet, so there is no
$S\bar\psi\psi$ at dimension $\le 4$. The model imposes $S \to -S$, which removes the odd terms
($S^3$ and $S\,H^\dagger H$). Then $S$ touches the SM only through $S^2 H^\dagger H$.

In [14]:
# ---- MOVE 4: check.  The portal count, and the trilinear left by the Z2 after <S> = v_S. ----
check_3 = cd.make_check_3(singlet)
answer_3 = len(step_3.portal_terms(singlet))
assert answer_3 == 2 and step_3.portal_terms(singlet, with_z2=True) == {"S S (H†H)": 4}
assert step_3.singlet_fermion_terms(singlet) == []
term, coeff = step_3.z2_trilinear(singlet)
display(Math(latex(term) + r"\;\xrightarrow{S\,\to\,v_S + S}\; " + latex(coeff) + r"\,S + \ldots"))
assert sp.simplify(coeff - 2 * singlet.extra["vS"].s * sp.expand(term / S.components[0]**2)) == 0
assert check_3(answer_3)
ok("two Higgs portals; with the Z2 only S^2 H^dag H, which gives 2 v_S S H^dag H after <S> = v_S")

<IPython.core.display.Math object>

[step_3] correct -- got 2
✓ two Higgs portals; with the Z2 only S^2 H^dag H, which gives 2 v_S S H^dag H after <S> = v_S


That trilinear is what makes the singlet mix with $h$ after both fields get a vacuum value: the
mass matrix of $(h, s)$ gets an off-diagonal entry $\lambda_{HS} v v_S$ *(the model's key result
`test_mass_matrix_eq7`)*.

**Beyond dimension 4.** A singlet can couple to gluons and photons through
$S\,G_{\mu\nu}G^{\mu\nu}/\Lambda_g$ and $S\,F_{\mu\nu}F^{\mu\nu}/\Lambda_\gamma$ (the latter from
$S\,B B$ and $S\,W W$), dimension 5. They come from loops of heavy charged or coloured states
*(cited: Franceschini et al., arXiv:1512.04933v3, Eq. (10); the photon operator from $S\,BB$ and
$S\,WW$ in Eqs. (13)–(14); the loops in Eqs. (20)–(21))*. The minimal
singlet model has no such states, so there $S$ reaches photons and gluons only through its
mixing with $h$. The record lists all four operators under `eft_operators`.

**The judgment to keep:** for a gauge singlet the lowest operator is fixed by the field content,
not chosen: $H^\dagger H$ is the only gauge-invariant SM combination of dimension below 3.

---
## 4. A singlet and a second doublet both put a scalar at 95 GeV, if you read the labels

Both candidates are built through `registry.build(model_id, benchmark=...)`, with the benchmark
solved here from physical targets. Each bundle must give the target masses back. The singlet's
mixing angle is set to an illustrative value, not fitted.

> **Before running the next cells.** The model rotates $(h_1, h_2) = R(\theta)\,(h, s)$, with
> $\theta$ from $\tan 2\theta$. With the singlet *lighter* than the Higgs boson, which label,
> $h_1$ or $h_2$, will the 95 GeV state get?

In [15]:
# ---- MOVE 1: set up.  The override: a 95 GeV singlet-like state under the measured Higgs. ----
from anomalies.higgs_95gev.solutions import step_4

m_heavy = step_4.heavy_mass(anomaly)
cand = step_4.build_singlet_candidate(anomaly)
bench = cand.benchmark
print(f"targets: {cd.TARGET_MASS_GEV} GeV (singlet-like) and {m_heavy} GeV (h-like, PDG 2024), "
      f"sin theta = {cd.SIN_THETA} (illustrative)")
print(f"solved inputs: MH = {bench['MH']:.2f} GeV, lamS = {bench['lamS']:.5f}, lamHS = {bench['lamHS']:.5f}")

targets: 95.0 GeV (singlet-like) and 125.2 GeV (h-like, PDG 2024), sin theta = 0.2 (illustrative)
solved inputs: MH = 124.13 GeV, lamS = 0.01858, lamHS = 0.01059


In [16]:
# ---- MOVE 2: collect.  The spectrum and the eigenvectors, from the bundle. ----
spec = step_4.singlet_spectrum(cand)
display(Math(r"M^2_{(h,s)} = " + latex(cand.extra["M_even"])))
print(f"MH1 = {spec['MH1']:.3f} GeV, MH2 = {spec['MH2']:.3f} GeV, theta = {spec['theta']:.4f}")
print(f"h2 composition: {spec['h2_singlet_fraction']:.2%} singlet, {spec['h2_doublet_fraction']:.2%} doublet")

<IPython.core.display.Math object>

MH1 = 125.200 GeV, MH2 = 95.000 GeV, theta = 0.2014
h2 composition: 96.00% singlet, 4.00% doublet


**Recognise.** The 95 GeV state is $h_2$. The principal branch of $\tan 2\theta$ keeps
$\theta$ small, so $h_1 = \cos\theta\,h + \sin\theta\,s$ is always the $h$-like state, whatever
the ordering. The model labels states **by mixing, not by mass**. Its labels are consistent,
since $h_2$ is mostly singlet, but its closed-form "$m^2_{1,2} = \ldots \mp \ldots$" assumes the
singlet is heavier *(computed; noted in UG-4)*.

In [17]:
# ---- MOVE 4: check.  The bundle re-derives the targets, and h2 is the singlet-like state. ----
check_4 = cd.make_check_4(cand)
answer_4 = spec["MH2"]
assert abs(spec["MH1"] - m_heavy) < 1e-6 and abs(math.sin(spec["theta"]) - cd.SIN_THETA) < 1e-9
assert spec["h2_singlet_fraction"] > spec["h2_doublet_fraction"]
assert check_4(answer_4)
ok(f"h2 = {answer_4:.2f} GeV, {spec['h2_singlet_fraction']:.0%} singlet; h1 = {spec['MH1']:.2f} GeV")

[step_4] correct -- got 95
✓ h2 = 95.00 GeV, 96% singlet; h1 = 125.20 GeV


Every $h_2$ coupling to SM particles is the SM one times $-\sin\theta$, because the model puts
$h = \cos\theta\,h_1 - \sin\theta\,h_2$ into every SM coupling of $h$ *(computed: the model's
key result `test_coupling_rescaling` pins $h_2WW$ and $h_2f\bar f$, and this record's
`test_h2_zz_scales_with_sin_theta` pins $h_2ZZ$)*. At 95 GeV $h_2$ cannot decay into $h_1$
pairs.

> **Before running the next cell.** What does that imply for $\mu_{\gamma\gamma}$,
> $\mu_{\tau\tau}$ and $\mu_{b\bar b}$ of $h_2$?

In [18]:
# ---- MOVE 3: recognise.  One number for every channel, next to the record; no verdict. ----
mu_pred = step_4.universal_mu(spec["theta"])
needed = step_4.sin2theta_needed(anomaly, [cd.step_2.DIPHOTON_ROW, cd.step_2.MU_TAUTAU_ROW])
rows = ["| channel | recorded $\\mu$ | $\\sin^2\\theta$ it would need | this benchmark |", "|---|---|---|---|"]
rows += [f"| {name.split('(')[1].split(',')[0]} | {obs[name].value:.2f} | {v:.2f} | {mu_pred:.3f} |"
         for name, v in needed.items()]
display(Markdown("\n".join(rows)))
assert abs(mu_pred - math.sin(spec["theta"]) ** 2) < 1e-12
ok(f"every h2 signal strength is sin^2 theta = {mu_pred:.3f} at this benchmark")

| channel | recorded $\mu$ | $\sin^2\theta$ it would need | this benchmark |
|---|---|---|---|
| CMS diphoton | 0.33 | 0.33 | 0.040 |
| CMS ditau | 1.37 | 1.37 | 0.040 |

✓ every h2 signal strength is sin^2 theta = 0.040 at this benchmark


**Read it, and stop there.** The singlet makes one prediction for every channel. Whether one
$\sin\theta$ fits both recorded values, within their errors and with the LHC limits on the
$h$-like state's $\cos\theta$, is the A3 fit, not a judgment made here.

**The second doublet.** `thdm_type2` would put a light CP-even $h$ at 95 GeV under a 125 GeV $H$.
It is CP-even because LEP's $e^+e^-\to Z\phi$ needs a $\phi ZZ$ coupling, and the CP-odd $A$ has
none *(physics judgment)*.

In [19]:
# ---- peek: the 2HDM at a Gunion-Haber-inverted point, and the labels the model gives back. ----
thdm, inv = step_4.build_thdm_candidate(anomaly)
t = step_4.thdm_spectrum(thdm)
p = step_4.THDM_POINT
print(f"target: h = {cd.TARGET_MASS_GEV} GeV under H = {m_heavy} GeV; tan beta = {p['tanb']}, "
      f"m_A = {p['mA']}, m_H+- = {p['mHp']} GeV")
print(f"alpha (Gunion-Haber) = {inv['alpha_GH']:.4f} > pi/4: {inv['alpha_GH'] > math.pi / 4}")
print(f"the model returns: MH0 (its h) = {t['MH0']:.2f} GeV, MHH (its H) = {t['MHH']:.2f} GeV, alpha = {t['alpha']:.4f}")
print(f"max |lambda_i| = {max(abs(inv[f'lam{k}']) for k in range(1, 6)):.1f}; "
      f"bounded from below: {step_4.thdm_bounded_from_below(inv)}")

target: h = 95.0 GeV under H = 125.2 GeV; tan beta = 2.0, m_A = 300.0, m_H+- = 600.0 GeV
alpha (Gunion-Haber) = 0.8025 > pi/4: True
the model returns: MH0 (its h) = 125.20 GeV, MHH (its H) = 95.00 GeV, alpha = -0.7683
max |lambda_i| = 11.8; bounded from below: True


**The trap, in the model this time.** The model's $\alpha$ is the principal branch of
$\tan 2\alpha$, which agrees with the convention $m_H \ge m_h$ only when $M_{11} > M_{22}$. Here
$\alpha > \pi/4$, so the model calls the 95 GeV state $H$ and the 125 GeV state $h$
*(computed)*. This is a labelling problem in feynlag-models, filed as UG-4
([`docs/upstream_gaps.md`](../../docs/upstream_gaps.md)), and a strict xfail test holds the
evidence. Nothing here works around it; §5 derives why it happens.

The swap needs $\alpha > \pi/4$, and $\alpha = \beta - \arcsin[\sin(\beta-\alpha)]$. Lower
$\tan\beta$ and the labels survive.

> **Before running the next cell.** At $\tan\beta = 1.5$, with the same $\sin(\beta-\alpha) = 0.3$,
> is $\alpha$ above or below $\pi/4$? Which state will the model call $h$?

In [20]:
# ---- MOVE 4: check.  A 2HDM point whose labels the model keeps, re-derived by the bundle. ----
thdm_ok, inv_ok = step_4.build_thdm_candidate(anomaly, step_4.THDM_ALIGNED_POINT)
t_ok = step_4.thdm_spectrum(thdm_ok)
q = step_4.THDM_ALIGNED_POINT
sin_bma = math.sin(math.atan(q["tanb"]) - t_ok["alpha"])
print(f"tan beta = {q['tanb']}, m_A = {q['mA']}, m_H+- = {q['mHp']} GeV, m12^2 = {q['m12sq']} GeV^2")
print(f"alpha (Gunion-Haber) = {inv_ok['alpha_GH']:.4f} < pi/4: {inv_ok['alpha_GH'] < math.pi / 4}")
print(f"the model returns: MH0 (its h) = {t_ok['MH0']:.2f} GeV, MHH (its H) = {t_ok['MHH']:.2f} GeV, "
      f"sin(beta - alpha) = {sin_bma:.3f}")
assert abs(t_ok["MH0"] - cd.TARGET_MASS_GEV) < 1e-6 and abs(t_ok["MHH"] - m_heavy) < 1e-6
assert abs(t_ok["MA0"] - q["mA"]) < 1e-6 and abs(t_ok["MHp"] - q["mHp"]) < 1e-6
assert abs(sin_bma - q["sin_bma"]) < 1e-9 and step_4.thdm_bounded_from_below(inv_ok)
worst = max(abs(inv_ok[f"lam{k}"]) for k in range(1, 6))
scan_min = min(r["max_abs_lambda"] for r in step_4.thdm_coupling_scan(
    thdm_ok.benchmark["v"], cd.TARGET_MASS_GEV, m_heavy))
print(f"max |lambda_i| = {worst:.1f} here; smallest over the bounded-from-below scan grid: "
      f"{scan_min:.2f}; 4 pi = {4 * math.pi:.1f}")
ok(f"labels kept: h = {t_ok['MH0']:.1f} GeV, H = {t_ok['MHH']:.1f} GeV, the physical point re-derived")

tan beta = 1.5, m_A = 100.0, m_H+- = 580.0 GeV, m12^2 = 8500.0 GeV^2
alpha (Gunion-Haber) = 0.6781 < pi/4: True
the model returns: MH0 (its h) = 95.00 GeV, MHH (its H) = 125.20 GeV, sin(beta - alpha) = 0.300
max |lambda_i| = 10.9 here; smallest over the bounded-from-below scan grid: 10.83; 4 pi = 12.6
✓ labels kept: h = 95.0 GeV, H = 125.2 GeV, the physical point re-derived


**Read it.** Below $\pi/4$ the model's $\alpha$ is Gunion–Haber's, so its $h$ is the 95 GeV
state with $\sin(\beta-\alpha) = 0.3$, and its $H$ is the 125 GeV one *(computed)*.
`thdm_type2` therefore has a usable point. UG-4 stays open for the other regime.

**A3 caveat, recorded without a verdict.** With $m_{H^\pm} \ge 580$ GeV *(cited:
Misiak–Steinhauser, through feynlag-models' `thdm_type2/NEXT_STEPS.md`)*, a light $h$ and $H$
need quartics near $4\pi$. That is $\max\lvert\lambda_i\rvert \approx 10.9$ here, and no
bounded-from-below point on the scan grid goes below 10.8 *(computed:
`step_4.thdm_coupling_scan`)*. Whether that is acceptable (perturbativity, unitarity) is
A3's question. The point also has a light $A$ at 100 GeV, chosen by the scan, not by data.

**The judgment to keep:** a benchmark override is only trustworthy once the bundle gives the
targets back *under the labels you will use*. Check the eigenvectors, not only the eigenvalues.

---
## 5. A mixing angle from $\tan 2\alpha$ is a convention plus a branch

Section 4 ran into the same thing twice. `sm_singlet_z2` puts its 95 GeV state in the second
slot, and `thdm_type2` calls a 95 GeV state $H$ once $\alpha > \pi/4$. The eigenvalues were right
both times *(computed, §4)*; what differs is which eigenvalue gets which name. This section shows
why on a plain real symmetric matrix, with no model attached, and then puts both candidates'
blocks through the same functions.

The rotation is feynlag-models' (`CONVENTIONS.md`, `rotation_2x2`): the new states are
$R(\alpha)$ times the old ones, with
$R(\alpha) = \begin{pmatrix} \cos\alpha & \sin\alpha \\ -\sin\alpha & \cos\alpha \end{pmatrix}$,
so the mass matrix of the new states is $R M R^T$.

> **Before running the next cell.** Setting the off-diagonal entry of $R M R^T$ to zero gives one
> equation for $\alpha$. How many angles in $(-\pi/2, \pi/2]$ solve it?

In [21]:
# ---- MOVE 1: set up.  A plain symmetric matrix and the rotation; no model, no Lagrangian. ----
from anomalies.higgs_95gev.solutions import step_5

a, b, d, M_gen = step_5.symmetric_2x2()
alpha = sp.Symbol("alpha", real=True)
display(Math(r"M = " + latex(M_gen) + r",\qquad R(\alpha) = " + latex(step_5.rotation(alpha))))

<IPython.core.display.Math object>

In [22]:
# ---- MOVE 2: collect.  The off-diagonal entry, its zero, and what alpha + pi/2 does. ----
off = step_5.offdiagonal(M_gen, alpha)
alpha_p = step_5.principal_angle(a, b, d)
display(Math(r"(R M R^T)_{12} = " + latex(off)))
display(Math(r"\alpha_{\rm principal} = " + latex(alpha_p)))
D_gen = step_5.rotated(M_gen, alpha)
D_shift = step_5.rotated(M_gen, alpha + sp.pi / 2)
assert sp.simplify(off.subs(alpha, alpha_p)) == 0
assert sp.simplify(D_shift[0, 1] + D_gen[0, 1]) == 0
assert sp.simplify(D_shift[0, 0] - D_gen[1, 1]) == 0 and sp.simplify(D_shift[1, 1] - D_gen[0, 0]) == 0
ok("the principal angle zeroes the off-diagonal entry; alpha + pi/2 does too, with the diagonal swapped")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ the principal angle zeroes the off-diagonal entry; alpha + pi/2 does too, with the diagonal swapped


**Recognise.** The condition is $\tan 2\alpha = 2b/(a-d)$. The tangent has period $\pi$, so
$2\alpha$ is fixed modulo $\pi$ and $\alpha$ only modulo $\pi/2$ *(computed)*. Two angles in
$(-\pi/2, \pi/2]$ solve it, and they differ in which state comes first: $R(\alpha + \pi/2)$ is
$R(\alpha)$ with its rows swapped and one of them negated.

Code has to pick one. feynlag's `diagonalize_orthogonal_2x2` takes the principal branch,
$\alpha = \tfrac12\arctan[2b/(a-d)]$, so $\lvert\alpha\rvert \le \pi/4$ and
$\lvert\cos\alpha\rvert \ge \lvert\sin\alpha\rvert$. The first rotated state is always the one
with the larger component along the first basis state *(computed)*. The principal branch labels
by **mixing**; nothing in it looks at which eigenvalue is larger.

> **Before running the next cell.** With $a > d$, does the first state get the larger eigenvalue
> or the smaller one? And with $a < d$?

In [23]:
# ---- MOVE 3: recognise.  The first diagonal entry on the principal branch, in each regime. ----
for sign, (computed, closed) in step_5.first_entry_by_regime().items():
    assert sp.simplify(computed - closed) == 0
    regime = r"a = d + \delta" if sign > 0 else r"a = d - \delta"
    display(Math(regime + r",\ \delta > 0:\qquad (R M R^T)_{11} = " + latex(closed)))
ok("first entry = (a + d)/2 + sign(a - d) sqrt((a - d)^2 + 4 b^2)/2 on the principal branch")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ first entry = (a + d)/2 + sign(a - d) sqrt((a - d)^2 + 4 b^2)/2 on the principal branch


**Recognise.** On the principal branch $\cos 2\alpha > 0$, so the first entry is
$\tfrac12\big[(a+d) + \mathrm{sign}(a-d)\sqrt{(a-d)^2 + 4b^2}\,\big]$: the larger eigenvalue when
$a > d$ and the smaller one when $a < d$ *(computed)*. A convention that says "the first label is
the lighter state", or "the heavier", is kept by the principal branch in one regime and broken in
the other. Now in numbers, one plain matrix per regime.

In [24]:
# ---- MOVE 4: check.  One matrix per regime, and the pi/2 shift that swaps the order. ----
for M_num in (step_5.DEMO_A_GT_D, step_5.DEMO_A_LT_D):
    r = step_5.branch_report(M_num)
    shift = step_5.mass_ordered_shift(M_num, first="lighter")
    D_num = step_5.rotated(sp.Matrix(M_num), r["alpha"] + shift).evalf()
    print(f"M = {M_num}: alpha = {r['alpha']:+.4f}, first = {r['first']:.4f}, second = {r['second']:.4f}, "
          f"first is lighter: {r['first_is_lighter']}, |cos alpha| = {r['overlap']:.3f}")
    print(f"    alpha {shift:+.4f} -> first = {float(D_num[0, 0]):.4f}, second = {float(D_num[1, 1]):.4f}")
    assert abs(r["offdiagonal"]) < 1e-12 and r["overlap"] >= 1 / math.sqrt(2)
    assert r["first_is_lighter"] == (M_num[0][0] < M_num[1][1])
    assert abs(float(D_num[0, 0]) - min(r["first"], r["second"])) < 1e-12
ok("principal branch: the first state is the heavier for a > d, the lighter for a < d; pi/2 swaps them")

M = ((3.0, 1.0), (1.0, 1.0)): alpha = +0.3927, first = 3.4142, second = 0.5858, first is lighter: False, |cos alpha| = 0.924
    alpha -1.5708 -> first = 0.5858, second = 3.4142
M = ((1.0, 1.0), (1.0, 3.0)): alpha = -0.3927, first = 0.5858, second = 3.4142, first is lighter: True, |cos alpha| = 0.924
    alpha +0.0000 -> first = 0.5858, second = 3.4142
✓ principal branch: the first state is the heavier for a > d, the lighter for a < d; pi/2 swaps them


**Now the candidates.** The functions above take any numeric block. Section 4 already built both
candidates' CP-even blocks: the singlet's $(h, s)$ block at the override, and the 2HDM's
$(\rho_1, \rho_2)$ block from the Gunion–Haber inversion, at both points. The conventions they are
held to, at the pinned feynlag-models `cb0678a`:

- `sm_singlet_z2` rotates $(h_1, h_2) = R(\theta)(h, s)$. Its key result writes
  $m^2_{1,2} = \lambda v^2 + \lambda_S v_S^2 \mp \sqrt{(\lambda v^2 - \lambda_S v_S^2)^2 + (\lambda_{HS} v v_S)^2}$,
  which makes $h_1$ the lighter state *(cited: `sm_singlet_z2/metadata.yaml`)*.
- `thdm_type2` rotates $(H, h) = R(\alpha)(\rho_1, \rho_2)$, and `CONVENTIONS.md:82` makes $h$ the
  lighter CP-even state *(cited)*. Gunion–Haber fix $\alpha$ from both $\sin 2\alpha$ and
  $\cos 2\alpha$ with $m_H \ge m_h$, and take $-\pi/2 \le \alpha \le \pi/2$ *(cited: Gunion–Haber,
  hep-ph/0207010, Eqs. (14)–(17))*. That range still allows $\alpha \to \alpha \pm \pi$, which is
  $h \to -h$, $H \to -H$ (their footnote 10): a field sign, not a relabelling.

> **Before running the next cell.** Is the singlet block at the override in the $a > d$ regime or
> the $a < d$ one? Which regime is the $\tan\beta = 2$ block in, and which the $\tan\beta = 1.5$
> one?

In [25]:
# ---- MOVE 4: check.  The same functions on the candidates' blocks, against their bundles. ----
M_s = step_5.singlet_mass_matrix(anomaly)
r_s = step_5.branch_report(M_s)
minus, plus = step_5.closed_form_roots(M_s)
print(f"singlet (h, s): M11 = {M_s[0, 0]:.0f}, M22 = {M_s[1, 1]:.0f} GeV^2 -> principal theta = "
      f"{r_s['alpha']:.4f} (model: {spec['theta']:.4f}), first slot (h1) = {math.sqrt(r_s['first']):.2f} GeV")
print(f"    key result, minus root (h1 lighter) = {math.sqrt(minus):.2f} GeV, plus root = {math.sqrt(plus):.2f} GeV")
assert M_s[0, 0] > M_s[1, 1] and not r_s["first_is_lighter"]
assert abs(r_s["alpha"] - spec["theta"]) < 1e-9 and abs(math.sqrt(r_s["first"]) - spec["MH1"]) < 1e-6
assert abs(math.sqrt(minus) - spec["MH2"]) < 1e-6
for name, inv_x, t_x in (("tan beta = 2", inv, t), ("tan beta = 1.5", inv_ok, t_ok)):
    r_x = step_5.branch_report(inv_x["M"])
    shift = step_5.mass_ordered_shift(inv_x["M"], first="heavier")
    print(f"2HDM, {name}: M11 - M22 = {inv_x['M'][0, 0] - inv_x['M'][1, 1]:+.0f} GeV^2 -> principal alpha = "
          f"{r_x['alpha']:+.4f} (model: {t_x['alpha']:+.4f}), first slot (H) = {math.sqrt(r_x['first']):.2f} GeV")
    print(f"    shift to put the heavier first: {shift / math.pi:+.2f} pi -> {r_x['alpha'] + shift:.4f} "
          f"= alpha_GH {inv_x['alpha_GH']:.4f}")
    assert abs(r_x["alpha"] - t_x["alpha"]) < 1e-9 and abs(math.sqrt(r_x["first"]) - t_x["MHH"]) < 1e-6
    assert abs(r_x["alpha"] + shift - inv_x["alpha_GH"]) < 1e-9
check_5 = cd.make_check_5(inv["M"])
answer_5 = step_5.mass_ordered_shift(inv["M"], first="heavier")
assert check_5(answer_5)
ok(f"singlet: a > d, h1 is the heavier; 2HDM: shift {answer_5 / math.pi:+.2f} pi at tan beta = 2, none at 1.5")

singlet (h, s): M11 = 15409, M22 = 9291 GeV^2 -> principal theta = 0.2014 (model: 0.2014), first slot (h1) = 125.20 GeV
    key result, minus root (h1 lighter) = 95.00 GeV, plus root = 125.20 GeV
2HDM, tan beta = 2: M11 - M22 = -227 GeV^2 -> principal alpha = -0.7683 (model: -0.7683), first slot (H) = 95.00 GeV
    shift to put the heavier first: +0.50 pi -> 0.8025 = alpha_GH 0.8025
2HDM, tan beta = 1.5: M11 - M22 = +1416 GeV^2 -> principal alpha = +0.6781 (model: +0.6781), first slot (H) = 125.20 GeV
    shift to put the heavier first: +0.00 pi -> 0.6781 = alpha_GH 0.6781
[step_5] correct -- got 1.5708
✓ singlet: a > d, h1 is the heavier; 2HDM: shift +0.50 pi at tan beta = 2, none at 1.5


**Read it.** All three blocks follow the rule derived above on a plain matrix *(computed)*.

- **Singlet.** At the override $M_{11} > M_{22}$: the $(h, h)$ entry is set by `MH`, just below
  the measured mass, and the singlet entry sits near 95 GeV. The principal $\theta$ puts the
  heavier, $h$-like state first, so $h_1$ is the 125 GeV state and the 95 GeV one is $h_2$. The
  key result's $\mp$ form, with $h_1$ on the minus root, holds in the other regime,
  $M_{11} < M_{22}$; here its minus root is $h_2$ *(computed)*. The labels agree with the model's
  rotation; only the closed form assumes a regime.
- **2HDM at $\tan\beta = 2$.** $M_{11} < M_{22}$, so the principal $\alpha$ puts the lighter state
  first, in the slot the model calls $H$, and Gunion–Haber's $\alpha$ is the principal one plus
  $\pi/2$ *(computed)*. The model's $h$ is then the heavier state, against `CONVENTIONS.md:82` at
  the pin. That is UG-4.
- **2HDM at $\tan\beta = 1.5$.** $M_{11} > M_{22}$: no shift, the principal branch is
  Gunion–Haber's, and the labels are kept *(computed)*.

Why $\tan\beta$ decides: the inversion builds $M = R^T \mathrm{diag}(m_H^2, m_h^2) R$, so
$M_{11} - M_{22} = (m_H^2 - m_h^2)\cos 2\alpha_{\rm GH}$, which changes sign at
$\alpha_{\rm GH} = \pi/4$ *(computed)*. At fixed $\sin(\beta - \alpha)$,
$\alpha_{\rm GH} = \beta - \arcsin[\sin(\beta - \alpha)]$ grows with $\tan\beta$.

The UG-4 strict xfail (`tests/test_step4_candidates.py::test_thdm_inverted_hierarchy_labels`) is
left as it is. This section explains it and works around nothing.

**When feynlag-models fixes UG-4** *(physics judgment)*. models-fn is working on the fix now, and
this record does not guess what it will choose. Every assert in §4 and in this section pins the
behaviour at the pinned feynlag-models `cb0678a`, so the pin bump that brings the fix will make
some of them fail. That is on purpose: the failures are the update list. Whatever labels the fix
chooses, the places that depend on the current ones are:

- the $\tan\beta = 2$ peek and its trap cell in §4, and the cell above where the principal
  $\alpha$ is compared with the model's;
- the strict xfail, which fails as soon as the labels follow mass; it then becomes an ordinary
  test, and UG-4 moves to Resolved in `docs/upstream_gaps.md`;
- the `candidate_models` notes in `anomaly.yaml`, `decisions.md` and the Recap rows;
- the singlet's statements here and in §4, if the fix also renames $h_1$ and $h_2$.

**The judgment to keep** *(physics judgment)*: a mixing angle read off $\tan 2\alpha$ is a
convention plus a branch. The equation fixes $\alpha$ only modulo $\pi/2$, the code's branch
decides which eigenvalue sits in which slot, and the convention decides what each slot is called.
Before using a model's $h$, $H$, $h_1$ or $h_2$, check its eigenvectors and eigenvalues against
the labels you will use: `step_5.branch_report` on the block, or the rotation in `bundle.extra`.

---
## 6. Recap

**The numbers**, each one produced by a cell above:

In [26]:
summary = [
    ("1", "the SM has one physical scalar after EWSB", f"{answer_1} ($m_h^2 = 2\\lambda v^2$; 4 dof − 3 Goldstones)", "computed"),
    ("1", "no SM scalar in 90–100 GeV", f"masses at benchmark: {', '.join(f'{m:.0f}' for m in masses)} GeV", "computed"),
    ("2", "SM-like $\\sigma_{\\rm ggF} B_{\\tau\\tau}$ at 95 GeV, 13 TeV", f"{denom:.2f} pb", "computed (inputs cited)"),
    ("2", "CMS ditau as a signal strength", f"$\\mu_{{\\tau\\tau}}$ = {answer_2:.2f} ± {exp_err:.2f} (exp) ± {th_err:.2f} (th)", "computed"),
    ("0", "CMS diphoton signal strength", f"$\\mu_{{\\gamma\\gamma}}$ = {gg.value:.2f} ± {gg.stat_uncertainty:.3f} (symmetrised)", "cited"),
    ("3", "operators coupling a singlet to the SM, dim ≤ 4", f"{answer_3}: $S H^\\dagger H$, $S^2 H^\\dagger H$; no $S\\bar\\psi\\psi$", "computed"),
    ("3", "dim-5 couplings to gluons and photons", "$S G G$, $S F F$, loop-induced", "cited"),
    ("4", "singlet-like state of `sm_singlet_z2` at the override", f"$h_2$ = {answer_4:.2f} GeV, {spec['h2_singlet_fraction']:.0%} singlet", "computed"),
    ("4", "universal signal strength of $h_2$", f"$\\sin^2\\theta$ = {mu_pred:.3f} (illustrative $\\theta$)", "computed"),
    ("4", "`thdm_type2` at an inverted point", f"labels swapped (its $H$ = {t['MHH']:.0f} GeV): UG-4", "computed"),
    ("4", "`thdm_type2` at $\\tan\\beta$ = 1.5", f"$h$ = {t_ok['MH0']:.0f}, $H$ = {t_ok['MHH']:.1f} GeV, labels kept; max $|\\lambda_i|$ = {worst:.1f}", "computed"),
    ("5", "a mixing angle from $\\tan 2\\alpha$", "fixed modulo $\\pi/2$; the principal branch labels by mixing", "computed"),
    ("5", "`thdm_type2` at $\\tan\\beta$ = 2, shift to mass order", f"{answer_5 / math.pi:+.2f} $\\pi$: its $h$ is the heavier state (UG-4)", "computed"),
]
rows = ["| § | result | key number | provenance |", "|---|---|---|---|"]
rows += [f"| {s} | {r} | {n} | {p} |" for s, r, n, p in summary]
display(Markdown("\n".join(rows)))

| § | result | key number | provenance |
|---|---|---|---|
| 1 | the SM has one physical scalar after EWSB | 1 ($m_h^2 = 2\lambda v^2$; 4 dof − 3 Goldstones) | computed |
| 1 | no SM scalar in 90–100 GeV | masses at benchmark: 125 GeV | computed |
| 2 | SM-like $\sigma_{\rm ggF} B_{\tau\tau}$ at 95 GeV, 13 TeV | 5.68 pb | computed (inputs cited) |
| 2 | CMS ditau as a signal strength | $\mu_{\tau\tau}$ = 1.37 ± 0.62 (exp) ± 0.17 (th) | computed |
| 0 | CMS diphoton signal strength | $\mu_{\gamma\gamma}$ = 0.33 ± 0.155 (symmetrised) | cited |
| 3 | operators coupling a singlet to the SM, dim ≤ 4 | 2: $S H^\dagger H$, $S^2 H^\dagger H$; no $S\bar\psi\psi$ | computed |
| 3 | dim-5 couplings to gluons and photons | $S G G$, $S F F$, loop-induced | cited |
| 4 | singlet-like state of `sm_singlet_z2` at the override | $h_2$ = 95.00 GeV, 96% singlet | computed |
| 4 | universal signal strength of $h_2$ | $\sin^2\theta$ = 0.040 (illustrative $\theta$) | computed |
| 4 | `thdm_type2` at an inverted point | labels swapped (its $H$ = 95 GeV): UG-4 | computed |
| 4 | `thdm_type2` at $\tan\beta$ = 1.5 | $h$ = 95, $H$ = 125.2 GeV, labels kept; max $|\lambda_i|$ = 10.9 | computed |
| 5 | a mixing angle from $\tan 2\alpha$ | fixed modulo $\pi/2$; the principal branch labels by mixing | computed |
| 5 | `thdm_type2` at $\tan\beta$ = 2, shift to mass order | +0.50 $\pi$: its $h$ is the heavier state (UG-4) | computed |

**The mechanics.**

0. The record holds what each experiment printed, in its own units: a ratio for diphoton, a cross
   section for ditau, nothing yet for LEP.
1. The SM from feynlag-models has one doublet: four real fields, three eaten by $W^\pm$, $Z$,
   one physical scalar with $m_h^2 = 2\lambda v^2$. Its mass is an input, so the SM predicts no
   second resonance anywhere, in particular none at 95 GeV. The protection is `field_content`.
2. Dividing the ditau $\sigma B$ by the SM-like $\sigma_{\rm ggF} B_{\tau\tau}$ at 95 GeV puts it
   on the diphoton scale, with a theory error kept apart from the experimental one.
3. feynlag's invariant scan on the singlet model's fields gives the Higgs portals
   $S H^\dagger H$ and $S^2 H^\dagger H$ and no Yukawa coupling of $S$. The model's $Z_2$ keeps
   only $S^2 H^\dagger H$, which still mixes $s$ with $h$ once $\langle S\rangle = v_S$.
4. A benchmark override solved from physical targets puts the singlet-like $h_2$ at 95 GeV, and
   every $h_2$ signal strength is $\sin^2\theta$. A 2HDM override with $\alpha < \pi/4$ keeps its
   labels; with $\alpha > \pi/4$ it comes back with $h$ and $H$ swapped (UG-4).
5. $\tan 2\alpha = 2b/(a-d)$ fixes a $2\times 2$ mixing angle only modulo $\pi/2$. The principal
   branch puts first the state closest to the first basis state, which is the heavier one when
   $M_{11} > M_{22}$ and the lighter one otherwise. That is why the singlet's 95 GeV state is
   $h_2$, and why `thdm_type2` swaps $h$ and $H$ at $\tan\beta = 2$ but not at 1.5.

**The tools, in the order you should reach for them.**

| question | tool |
|---|---|
| how many physical scalars? | `bundle.model.mass_matrix` on the neutral and charged blocks, then count nonzero eigenvalues |
| what is their mass? | the same eigenvalues, `subs(bundle.values())` at the benchmark |
| a channel in pb, the others in $\mu$? | divide by the SM-like $\sigma \times B$ from the LHC Higgs WG tables (`solutions/step_2.py`) |
| inputs consumed by code | `anomaly.sm_prediction.sm_reference_values`, rejected if `TODO_VERIFY` |
| which operators can a new field have? | `feynlag.suggest_potential`, `feynlag.suggest_yukawa` on the model's own fields |
| a model point from physical masses | solve the benchmark, `registry.build(id, benchmark=...)`, read the spectrum back |
| which state is which? | the rotation's eigenvectors (`bundle.extra["rot"]`), not the order of the eigenvalues |
| is my angle the one my labels assume? | `step_5.branch_report` on the block: the sign of $M_{11} - M_{22}$ and the shift to mass order |

**The traps**, each of which a cell above turned into a check or a caveat:

- **The $Z$ is in the window.** It is spin 1, cannot give a diphoton peak (Landau–Yang), and is
  already part of the background.
- **"SM-like" is a hypothetical boson**, the SM Higgs with its mass moved to 95 GeV. It is a
  normalisation, not an SM prediction for a signal: the SM predicts zero.
- **Equal units do not mean comparable channels.** Production and decay differ between ditau and
  diphoton.
- **Two error types.** Experimental and theory normalisation errors are kept apart for the fit.
- **Labels follow mixing, not mass.** In `sm_singlet_z2` the 95 GeV state is $h_2$. In
  `thdm_type2` the principal-branch $\alpha$ swaps $h$ and $H$ when $M_{11} < M_{22}$. §5 shows
  why: the principal branch of $\tan 2\alpha$ labels by mixing, never by mass.
- **`MH` is not a mass eigenvalue** in the singlet model. It fixes the $(h,h)$ entry, so the
  override needs `MH` below the measured 125.20 GeV.

**Next steps (A3).** A stage-1 $\chi^2$ of the singlet's $\sin^2\theta$ against
$\mu_{\gamma\gamma}$ and $\mu_{\tau\tau}$ (one ditau row only, `EXCLUSIVE_WITH`), with the LHC
limits on the $h$-like state's $\cos\theta$ and, once found in a primary source, the LEP
$b\bar b$ signal strength. For the 2HDM at the $\tan\beta = 1.5$ point, A3 must first settle
the quartics near $4\pi$. Its $\mu_{\gamma\gamma}$ needs the $H^\pm$ loop in $h\to\gamma\gamma$,
which feynlag does not compute yet. UG-4 has to be fixed before points with $\alpha > \pi/4$
can be used. The pin bump that brings that fix will fail asserts in §4 and §5 on purpose; §5 lists
what to update then.